In [1]:
import sys
sys.path.append('../src/')

from dataset import gee_data as gd
from dataset import feature_extraction as fe
from dataset import glamos_processing as glamos
import ee

In [2]:
gdf = glamos.get_data(2000, 2025)

Extracting glacier mass balance...
Extracting glacier geometry...


In [3]:
gd.initialize_gee('ee-jaybrandon-dspro2')

In [4]:
def assign_satellite_label(date):
    year = date.year
    if year < 1984:
        return None
    elif 1984 <= year < 2013:
        return "landsat5"
    elif 2013 <= year <= 2016:
        return "landsat8"
    else:
        return "sentinel2"

In [5]:
if gdf.crs != "EPSG:4326":
    gdf = gdf.to_crs(epsg=4326)
gdf["geometry"] = gdf.geometry.buffer(0)
gdf["satellite"] = gdf["observation_end"].apply(assign_satellite_label)

In [6]:
gdf

,geometry,id,observation_start,observation_end,mass_balance_annual,coordx,coordy,obs_id,satellite
0,"POLYGON ((9.97176 46.7677, 9.97185 46.76765, 9...",A10g-18,2006-10-01,2007-09-30,-855,2793104,1182537,A10g-18_2006-10-01_2007-09-30,landsat5
1,"POLYGON ((9.97176 46.7677, 9.97185 46.76765, 9...",A10g-18,2007-10-01,2008-09-30,-554,2793104,1182537,A10g-18_2007-10-01_2008-09-30,landsat5
2,"POLYGON ((9.97176 46.7677, 9.97185 46.76765, 9...",A10g-18,2008-10-01,2009-09-30,-724,2793104,1182537,A10g-18_2008-10-01_2009-09-30,landsat5
3,"POLYGON ((9.97176 46.7677, 9.97185 46.76765, 9...",A10g-18,2009-10-01,2010-09-30,-574,2793104,1182537,A10g-18_2009-10-01_2010-09-30,landsat5
4,"POLYGON ((9.97176 46.7677, 9.97185 46.76765, 9...",A10g-18,2010-10-01,2011-09-30,-1116,2793104,1182537,A10g-18_2010-10-01_2011-09-30,landsat5
...,...,...,...,...,...,...,...,...,...
518,"POLYGON ((10.0934 46.84986, 10.09289 46.84922,...",A10g-05,2020-10-01,2021-09-30,-888,2801722,1192164,A10g-05_2020-10-01_2021-09-30,sentinel2
519,"POLYGON ((10.0934 46.84986, 10.09289 46.84922,...",A10g-05,2021-10-01,2022-09-30,-3339,2801722,1192164,A10g-05_2021-10-01_2022-09-30,sentinel2
520,"POLYGON ((10.0934 46.84986, 10.09289 46.84922,...",A10g-05,2022-10-01,2023-09-30,-2309,2801722,1192164,A10g-05_2022-10-01_2023-09-30,sentinel2
521,"POLYGON ((10.0934 46.84986, 10.09289 46.84922,...",A10g-05,2023-10-01,2024-09-30,-1654,2801722,1192164,A10g-05_2023-10-01_2024-09-30,sentinel2


In [7]:
row = gdf.iloc[0]

In [8]:
row

geometry               POLYGON ((9.971755098542367 46.7677014664532, ...
id                                                               A10g-18
observation_start                                    2006-10-01 00:00:00
observation_end                                      2007-09-30 00:00:00
mass_balance_annual                                                 -855
coordx                                                           2793104
coordy                                                           1182537
obs_id                                     A10g-18_2006-10-01_2007-09-30
satellite                                                       landsat5
Name: 0, dtype: object

In [9]:
roi = ee.Geometry(row.geometry.__geo_interface__)

collection = gd.get_glacier_collection(
    sensor_type=row['satellite'],
    polygon=roi,
    start_date=row["observation_start"].strftime("%Y-%m-%d"),
    end_date=row["observation_end"].strftime("%Y-%m-%d"),
    cloud_threshold=40,
)

In [10]:
collection

In [11]:
def add_date(img):
        date_str = img.date().format('YYYY-MM-DD')
        return img.set('date', date_str)
    
col_with_date = collection.map(add_date)

In [12]:
l1 = col_with_date.aggregate_histogram('date').keys()

In [13]:
l2 = col_with_date.aggregate_array('date').distinct()

In [14]:
col_with_date.propertyNames().getInfo()

[]

In [15]:
l2.getInfo()

['2007-08-238', '2007-09-254', '2007-09-245']

In [16]:
l1.getInfo()

['2007-08-238', '2007-09-245', '2007-09-254']

In [17]:
dem = gd.get_dem(roi)

In [18]:
results = fe.extract_glacier_period_features(collection, dem, roi, row['obs_id'])

In [19]:
results

{'features': [{'type': 'Feature',
   'geometry': None,
   'id': '0',
   'properties': {'B11': 0.062198539937759326,
    'B12': 0.04684766856846473,
    'B2': 0.08741365923236515,
    'B3': 0.10569209024896264,
    'B4': 0.10368610736514522,
    'B8': 0.1027907183609958,
    'NDSI': 0.30956465847308084,
    'area_m2': 1233.436767578125,
    'aspect_mean': 263,
    'date': '2007-08-238',
    'elev_mean': 2786.177889823104,
    'obs_id': 'A10g-18_2006-10-01_2007-09-30',
    'sla': 2719.07958984375,
    'slope_mean': 1,
    'snow_fraction': 0.04809505846850245}},
  {'type': 'Feature',
   'geometry': None,
   'id': '1',
   'properties': {'B11': None,
    'B12': None,
    'B2': None,
    'B3': None,
    'B4': None,
    'B8': None,
    'NDSI': None,
    'area_m2': 0,
    'aspect_mean': 263,
    'date': '2007-09-254',
    'elev_mean': 2786.177889823104,
    'obs_id': 'A10g-18_2006-10-01_2007-09-30',
    'sla': None,
    'slope_mean': 1,
    'snow_fraction': None}},
  {'type': 'Feature',
   'ge

In [20]:
results.get("final_mask_image")

In [21]:
results.get("features")[0]

{'type': 'Feature',
 'geometry': None,
 'id': '0',
 'properties': {'B11': 0.062198539937759326,
  'B12': 0.04684766856846473,
  'B2': 0.08741365923236515,
  'B3': 0.10569209024896264,
  'B4': 0.10368610736514522,
  'B8': 0.1027907183609958,
  'NDSI': 0.30956465847308084,
  'area_m2': 1233.436767578125,
  'aspect_mean': 263,
  'date': '2007-08-238',
  'elev_mean': 2786.177889823104,
  'obs_id': 'A10g-18_2006-10-01_2007-09-30',
  'sla': 2719.07958984375,
  'slope_mean': 1,
  'snow_fraction': 0.04809505846850245}}